<a href="https://colab.research.google.com/github/Tinnnn-vn/langchain-guide/blob/main/rag_basic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! pip install langchain-community
! pip install wikipedia-api
! pip install langchain
! pip install langchain-text-splitters
! pip install langchain-openai
! pip install langchain-chroma

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.2/129.2 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 7.5 MB/s eta 0:00:00
  Attempting u

# Document Ingestion

In [ ]:
import wikipediaapi

wiki = wikipediaapi.Wikipedia(
    user_agent='MyDataPipelineBot/1.0 (contact@example.com)',
    language='vi'
)

page = wiki.page('Tesla, Inc.')

if page.exists():
    print("Title:", page.title)
    print("\nSummary:", page.summary[:200])
    print("\nFull Text length:", len(page.text))
else:
    print("Page does not exist.")


Title: Tesla, Inc.

Summary: Tesla, Inc. là một công ty chuyên về sản xuất ô tô điện của Hoa Kỳ được thành lập vào năm 2003. Đây là một công ty đại chúng và giao dịch trên sàn chứng khoán NASDAQ dưới mã TSLA. Trong quý đầu năm 20

Full Text length: 49013


In [ ]:
# In toàn bộ nội dung
if page.exists():
    print("TOÀN BỘ NỘI DUNG BÀI VIẾT")
    print(page.text)
else:
    print("Bài viết không tồn tại.")

=== TOÀN BỘ NỘI DUNG BÀI VIẾT ===
Tesla, Inc. là một công ty chuyên về sản xuất ô tô điện của Hoa Kỳ được thành lập vào năm 2003. Đây là một công ty đại chúng và giao dịch trên sàn chứng khoán NASDAQ dưới mã TSLA. Trong quý đầu năm 2013, Tesla lần đầu tiên công bố lợi nhuận sau 10 năm phát triển.
Tesla lần đầu gây chú ý bằng việc ra mắt mẫu xe Tesla Roadster, chiếc xe thể thao đầu tiên chạy hoàn toàn bằng điện. Dòng xe thứ hai của công ty là mẫu Model S, một chiếc sedan hạng sang khác cũng chạy bằng năng lượng điện.
Tesla cũng cung cấp linh kiện cho xe điện của các hãng chế tạo ô tô khác như Daimler và Toyota. Giám đốc điều hành của Tesla, Elon Musk, cho biết ông hy vọng Tesla sẽ là một hãng chế tạo ô tô độc lập, với mục tiêu đem đến những chiếc xe điện có giá cả phù hợp cho người tiêu dùng bình dân.

Khái quát
Tesla được đặt theo tên kỹ sư điện, nhà phát minh Nikola Tesla. Chiếc Tesla Roadster sử dụng mô tơ điện xoay chiều dựa trực tiếp trên thiết kế gốc năm 1882 của Tesla. Tesla Road

# Text chunking
Khi text_splitter cắt xong Chunk 0, nó sẽ nhìn vào cấu hình chunk_overlap=200. Lúc này, nó sẽ lùi lại tối đa khoảng 200 ký tự tính từ điểm cắt của Chunk 0 để làm "vùng tìm kiếm" cho điểm bắt đầu của Chunk 1.

Trong vùng 200 ký tự đó, nó vẫn sẽ cố gắng tìm một khoảng trắng " " (khoảng cách giữa các từ) thích hợp nhất để cắt. Do đó, đoạn lặp lại thực tế có thể là 185 hoặc 192 ký tự (để từ ngữ không bị sứt mẻ), chứ hiếm khi đúng 200 ký tự.

chunk_overlab hữu ích khi đoạn văn bản chứa các ký tự liên tục, và chunk 1 sẽ lặp lại 200 ký tự cuối của chunk 0.

Khi gặp dấu xuống dòng \n trong vùng tìm kiếm, thuật toán sẽ nhảy ngay đến sau dấu \n đó để bắt đầu Chunk 1. Vì vậy, hiện tượng lặp lại chữ sẽ không xảy ra nữa.

Nếu muốn cắt văn bản dựa trên ngữ nghĩa, bạn phải sử dụng một bộ chia nâng cao hơn của LangChain gọi là SemanticChunker.

Tuy nhiên, các nghiên cứu điểm chuẩn (benchmark) cho thấy Semantic Chunking không phải lúc nào cũng mang lại hiệu quả vượt trội như lý thuyết. Trong các tài liệu có cấu trúc tốt (như Wikipedia hay sách giáo khoa), phương pháp cắt đếm ký tự thông thường (RecursiveCharacterTextSplitter) đạt độ chính xác gần như tương đương, thậm chí tốt hơn do kiểm soát được kích thước chunk ổn định.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Khởi tạo bộ tách văn bản
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
)

# Chia văn bản thành từng đoạn nhỏ.
chunks = text_splitter.create_documents([page.text])

# In 2 chunk đầu tiên
print(f"chunk 0: {chunks[0].page_content}")
print(f"chunk 1: {chunks[1].page_content}")


chunk 0: Tesla, Inc. là một công ty chuyên về sản xuất ô tô điện của Hoa Kỳ được thành lập vào năm 2003. Đây là một công ty đại chúng và giao dịch trên sàn chứng khoán NASDAQ dưới mã TSLA. Trong quý đầu năm 2013, Tesla lần đầu tiên công bố lợi nhuận sau 10 năm phát triển.
Tesla lần đầu gây chú ý bằng việc ra mắt mẫu xe Tesla Roadster, chiếc xe thể thao đầu tiên chạy hoàn toàn bằng điện. Dòng xe thứ hai của công ty là mẫu Model S, một chiếc sedan hạng sang khác cũng chạy bằng năng lượng điện.
Tesla cũng cung cấp linh kiện cho xe điện của các hãng chế tạo ô tô khác như Daimler và Toyota. Giám đốc điều hành của Tesla, Elon Musk, cho biết ông hy vọng Tesla sẽ là một hãng chế tạo ô tô độc lập, với mục tiêu đem đến những chiếc xe điện có giá cả phù hợp cho người tiêu dùng bình dân.
chunk 1: Khái quát
Tesla được đặt theo tên kỹ sư điện, nhà phát minh Nikola Tesla. Chiếc Tesla Roadster sử dụng mô tơ điện xoay chiều dựa trực tiếp trên thiết kế gốc năm 1882 của Tesla. Tesla Roadster, mẫu xe đầu 

# Embedding

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "your_api_key"

In [ ]:
from langchain_openai import OpenAIEmbeddings

# Khởi tạo model embeddings
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-large"
)
# Trích xuất phần chữ từ biến chunks ở đoạn mã trước đó
chunk_texts = [doc.page_content for doc in chunks]

# Gửi trực tiếp danh sách chữ này lên OpenAI để lấy vector
print("Đang tạo Vector Embedding từ các chunks có sẵn...")
vectors = embeddings_model.embed_documents(chunk_texts)
# Các chunks văn bản bây giờ là các vector toán học
print(f"Đã chuyển đổi thành công {len(vectors)} chunk thành các Vector toán học!")


Đang tạo Vector Embedding từ các chunks có sẵn...
Đã chuyển đổi thành công 75 chunk thành các Vector toán học!


# Vector database

Kho lưu trữ vector là một hệ thống lưu trữ được thiết kế để chứa các vector nhúng (vector embeddings) - vốn là các biểu diễn đa chiều của dữ liệu.

Mục đích: Cho phép thực hiện tìm kiếm độ tương đồng nhanh chóng bằng cách so sánh các vectơ truy vấn với các vectơ tài liệu đã lưu trữ để tìm ra những kết quả phù hợp nhất.

In [ ]:
from langchain_chroma import Chroma

# Khởi tạo kho lưu trữ Chroma trong bộ nhớ tạm
# Thao tác này sẽ tự động gửi dữ liệu qua OpenAIEmbeddings để lấy vector và nạp vào DB
print("Đang mã hóa và lưu dữ liệu vào ChromaDB...")

db = Chroma.from_documents(
    documents=chunks,                # Biến chunks từ bước trước
    embedding=embeddings_model       # Mô hình OpenAIEmbeddings đã khởi tạo ở bước trước
)
print("Đã lưu toàn bộ dữ liệu vào ChromaDB thành công!")

Đang mã hóa và lưu dữ liệu vào ChromaDB...
Đã lưu toàn bộ dữ liệu vào ChromaDB thành công!


# Retrieval

In [ ]:
# Chuyển đổi Vector DB thành một Retriever
# search_type="similarity": Tìm kiếm dựa trên độ tương đồng ngữ nghĩa
# search_kwargs={"k": 2}: Chỉ lấy ra 2 đoạn văn bản có điểm tương quan cao nhất
retriever = db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

# Tạo một câu hỏi giả định để kiểm tra bộ truy xuất
user_question = "Ai đã tạo ra công ty xe điện của Elon Musk?"

# Tiến hành gọi Retriever để tìm kiếm dữ liệu (Đây chính là hành động Retrieval)
retrieved_docs = retriever.invoke(user_question)

# Hiển thị kết quả truy xuất được
print(f"🎯 Bộ lọc Retrieval đã tìm thấy {len(retrieved_docs)} đoạn tài liệu phù hợp.")
print("=" * 60)

for index, doc in enumerate(retrieved_docs):
    print(f"📌 Đoạn văn bản thứ {index + 1}:")
    print(doc.page_content)
    print("-" * 60)

🎯 Bộ lọc Retrieval đã tìm thấy 2 đoạn tài liệu phù hợp.
📌 Đoạn văn bản thứ 1:
Việc sáng lập công ty này là một vấn đề tranh tụng sau đó đã được xử lý sau một thỏa thuận bên ngoài tòa án. 26/5/2009, Eberhard đã đệ trình lên San Mateo County, California, kiện Tesla và Musk (Chủ tịch kiếm Giám đốc điều hành của Tesla) vì phỉ báng, phá vỡ hợp đồng. Musk đã viết một post dài trên blog trong đó đưa ra những tài liệu gốc ban đầu, các email giữa các giám đốc cấp cao và các hiện vật chứng minh khác chỉ ra rằng Eberhard đã bị sa thải bởi sự nhất trí của ban lãnh đạo Tesla. 29/6/2009, một phiên tòa ở San Mateo County, California, tòa án tối cao đã bác bỏ các đòi hỏi của cựu CEO Eberhard, người đã yêu cầu phải được tuyên bố là một trong 2 người sáng lập công ty. Tesla đã phát biểu trong một thông báo rằng quy tăc là "phù hợp với sự tin tưởng của Tesla trong đội ngũ sáng lập, bao gồm CEO và kiến trúc sư trưởng hiện tại Elon Musk, giám đốc công nghệ JB Straubel, những người đặt nền móng cho việc tạo

# Multi-query Retrieval

Multi-query Retrieval là một kỹ thuật nâng cao rất hay trong RAG để giải quyết một điểm yếu chí mạng của tìm kiếm vector thông thường: Người dùng gõ câu hỏi quá ngắn hoặc dùng từ ngữ không giống với tài liệu gốc.

Nếu tài liệu viết: "Tesla được thành lập bởi Martin Eberhard..."Nhưng người dùng hỏi: "Ai đã tạo ra công ty xe điện của Elon Musk?"Hệ thống tìm kiếm vector thông thường (Single Query) có thể tìm trượt vì câu hỏi không chứa từ "Tesla" hay "thành lập".

Giải pháp của Multi-query Retrieval:

Khi bật tính năng này, quy trình sẽ diễn ra như sau:

- Bạn gửi câu hỏi gốc vào hệ thống.
- Hệ thống đưa câu hỏi này cho một Mô hình ngôn ngữ lớn (LLM) và bảo nó: "Hãy viết lại câu hỏi này thành 3 hoặc 4 câu khác nhau có cùng ý nghĩa bằng các từ đồng nghĩa".
- Hệ thống mang cả 4 câu hỏi này đi quét đồng thời (Multi-query) vào Vector DB để nhặt tài liệu.Nó gộp tất cả kết quả lại, loại bỏ các đoạn trùng lặp để đưa ra tập tài liệu chính xác nhất.

NÊN DÙNG Multi-query: Khi làm hệ thống Chatbot chăm sóc khách hàng, tư vấn nội bộ – nơi mà độ chính xác của câu trả lời là ưu tiên số một, và người dùng thường hỏi một cách mơ hồ.

KHÔNG NÊN DÙNG: Khi hệ thống yêu cầu tốc độ phản hồi cực cao (Real-time), ngân sách vận hành API hạn hẹp, hoặc dữ liệu đầu vào đã được cấu hình từ khóa chuẩn hóa 100%.

Trong đoạn code dưới bước Augmentation (hay còn gọi là làm giàu prompt) chỉ là ghép câu hỏi của người dùng (prompt) và dữ liệu văn bản có liên quan được trích xuất từ tài liệu (chunk) -> sau đó tạo thành prompt giàu ngữ nghĩa rồi đưa cho llm trả lời

In [ ]:
! pip install langchain-classic

In [ ]:
import logging
import sys
from google.colab import userdata
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# CẤU HÌNH HỆ THỐNG & LOGGER (ÉP HIỂN THỊ CÂU HỎI PHỤ)
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)


# KHỞI TẠO CÁC THÀNH PHẦN AI & RETRIEVER
# Mô hình LLM dùng để sinh câu hỏi phụ và trả lời câu hỏi cuối cùng
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Kế thừa cơ sở dữ liệu ChromaDB từ các bước trước
# (Yêu cầu ô mã tạo biến 'db' ở bước trước đã được chạy thành công)
base_retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 2})

# Đóng gói thành bộ truy xuất Multi-query nâng cao
advanced_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm
)

# ĐỊNH NGHĨA PROMPT (KHUNG HƯỚNG DẪN CHO AI)
template = """Bạn là một trợ lý thông minh. Hãy trả lời câu hỏi dựa VÀO CHÍNH XÁC nội dung tài liệu ngữ cảnh được cung cấp dưới đây. Nếu tài liệu không có thông tin, hãy nói "Tôi không biết".

Tài liệu ngữ cảnh:
{context}

Câu hỏi: {question}
Câu trả lời:"""

prompt = ChatPromptTemplate.from_template(template)

# THỰC THI QUY TRÌNH MULTI-QUERY VÀ IN KẾT QUẢ
# Câu hỏi đánh đố, không chứa từ khoá trực diện "Tesla", "Nikola"
complex_question = "Nguồn gốc tên gọi của thương hiệu xe điện này bắt nguồn từ ai?"

print("--- [BƯỚC 1] QUÁ TRÌNH MULTI-QUERY GENERATION ---")
print("Hệ thống đang tự sinh các câu hỏi phụ (Xem ở dòng INFO phía dưới):\n")

# Kích hoạt bộ truy xuất (Hệ thống sẽ in ra các câu hỏi phụ ở bước này nhờ Logger)
retrieved_docs = advanced_retriever.invoke(complex_question)

print("\n--- [BƯỚC 2] Augmentation - TỔNG HỢP VÀ SINH CÂU TRẢ LỜI ---")
# Gộp các đoạn tài liệu tìm được thành một chuỗi văn bản lớn
context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])

# Thiết lập chuỗi xử lý sinh câu trả lời (Generation Chain)
generation_chain = prompt | llm | StrOutputParser()

# Gọi AI tạo câu trả lời hoàn chỉnh dựa trên ngữ cảnh đã "bốc" được
final_answer = generation_chain.invoke({
    "context": context_text,
    "question": complex_question
})

print(f"\n❓ Câu hỏi của bạn: '{complex_question}'")
print("-" * 50)
print("🤖 Câu trả lời hoàn chỉnh từ AI:")
print(final_answer)


INFO:langchain_openai.chat_models._client_utils:langchain-openai detected system proxy configuration and no explicit `http_socket_options` / `http_client` / `http_async_client` / `openai_proxy`; skipping the custom `httpx` transport so httpx's env-proxy auto-detection applies. Pass `http_socket_options=[...]` to opt back into kernel-level TCP keepalive tuning on top of the env proxy.
--- [BƯỚC 1] QUÁ TRÌNH MULTI-QUERY GENERATION ---
Hệ thống đang tự sinh các câu hỏi phụ (Xem ở dòng INFO phía dưới):

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['1. Ai là người sáng lập hoặc có ảnh hưởng lớn đến tên gọi của thương hiệu xe điện này?  ', '2. Tên gọi của thương hiệu xe điện này có ý nghĩa gì và ai đã đặt tên cho nó?  ', '3. Lịch sử và nguồn gốc của tên thương hiệu xe điện này là gì, và ai là người đứng sau nó?']
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "H

# Generation

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Thiết kế Prompt (Khung hướng dẫn tối ưu cho Tiếng Việt)
template = """Bạn là một chuyên gia AI được đào tạo để trả lời dựa trên tài liệu.
Hãy đọc kỹ tài liệu ngữ cảnh dưới đây và trả lời câu hỏi một cách chính xác, khách quan.
Nếu tài liệu không chứa thông tin để trả lời, hãy nói thẳng thắn "Tôi không biết dựa trên dữ liệu hiện có". Không tự bịa thông tin.

Tài liệu ngữ cảnh:
{context}

Câu hỏi: {question}
Câu trả lời của chuyên gia:"""

prompt = ChatPromptTemplate.from_template(template)

# Hàm hỗ trợ định dạng: Gộp toàn bộ các đoạn văn bản nhặt được thành 1 chuỗi lớn
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

# 3. Tạo chuỗi RAG hoàn chỉnh (Tự động hóa hoàn toàn)
# - Nhận câu hỏi đầu vào
# - Chạy qua advanced_retriever (đã có Multi-query) để lấy tài liệu
# - Đưa tài liệu vào prompt cùng câu hỏi gốc
# - Gửi cho llm xử lý và parse kết quả đầu ra thành dạng text thuần
rag_chain = (
    {
        "context": advanced_retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# 4. Kiểm tra thử nghiệm hệ thống cuối cùng
user_query = "Thương hiệu xe này đặt tên theo Nikola Tesla đúng không?"
print("⏳ Hệ thống đang quét dữ liệu nâng cao và suy luận câu trả lời...\n")

response = rag_chain.invoke(user_query)

print("=" * 60)
print(f"❓ Câu hỏi từ người dùng: {user_query}")
print("=" * 60)
print("🤖 Chatbot RAG trả lời:")
print(response)
print("=" * 60)

"""
Có thể sử dụng stream() thay cho invoke() để in ra từng token một (như ChatGPT) mà không cần phải chờ đợi kết quả.
for chunk in response:
    print(chunk, end="", flush=True)

print("\n" + "=" * 60)
"""


⏳ Hệ thống đang quét dữ liệu nâng cao và suy luận câu trả lời...

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['1. Xe này có phải được đặt tên theo Nikola Tesla không?  ', '2. Thương hiệu xe này có liên quan đến Nikola Tesla không?  ', '3. Có phải tên của thương hiệu xe này lấy cảm hứng từ Nikola Tesla?']
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
❓ Câu hỏi từ người dùng: Thương hiệu xe này đặt tên theo Nikola Tesla đúng không?
🤖 Chatbot RAG trả lời:
Đúng, thương hiệu xe Tesla được đặt theo tên kỹ sư điện, nhà phát minh Nikola Tesla.


' Có thể sử dụng stream() thay cho invoke() để in ra từng token một (như ChatGPT) mà không cần phải chờ đợi kết quả.\nfor chunk in response:\n    print(chunk, end="", flush=True)\n\nprint("\n" + "=" * 60)\n'